In [63]:
import timm

In [64]:
timm.list_models()

['aimv2_1b_patch14_224',
 'aimv2_1b_patch14_336',
 'aimv2_1b_patch14_448',
 'aimv2_3b_patch14_224',
 'aimv2_3b_patch14_336',
 'aimv2_3b_patch14_448',
 'aimv2_huge_patch14_224',
 'aimv2_huge_patch14_336',
 'aimv2_huge_patch14_448',
 'aimv2_large_patch14_224',
 'aimv2_large_patch14_336',
 'aimv2_large_patch14_448',
 'bat_resnext26ts',
 'beit3_base_patch16_224',
 'beit3_giant_patch14_224',
 'beit3_giant_patch14_336',
 'beit3_large_patch16_224',
 'beit_base_patch16_224',
 'beit_base_patch16_384',
 'beit_large_patch16_224',
 'beit_large_patch16_384',
 'beit_large_patch16_512',
 'beitv2_base_patch16_224',
 'beitv2_large_patch16_224',
 'botnet26t_256',
 'botnet50ts_256',
 'caformer_b36',
 'caformer_m36',
 'caformer_s18',
 'caformer_s36',
 'cait_m36_384',
 'cait_m48_448',
 'cait_s24_224',
 'cait_s24_384',
 'cait_s36_384',
 'cait_xs24_384',
 'cait_xxs24_224',
 'cait_xxs24_384',
 'cait_xxs36_224',
 'cait_xxs36_384',
 'coat_lite_medium',
 'coat_lite_medium_384',
 'coat_lite_mini',
 'coat_lite_sma

# Test Inferance

In [65]:
from whoot_model_training.whoot_model_training.data_extractor import raw_audio_extractor


ds = raw_audio_extractor("data/2023/Otay/5_seconds")
ds

Resolving data files:   0%|          | 0/31537 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'labels'],
        num_rows: 31537
    })
    valid: Dataset({
        features: ['audio', 'labels'],
        num_rows: 31537
    })
    test: Dataset({
        features: ['audio', 'labels'],
        num_rows: 31537
    })
})

In [66]:
ds["train"][0]

{'audio': {'path': '/home/sean/whoot/data/2023/Otay/5_seconds/20230607_000001_part1.wav',
  'array': array([ 0.02090454,  0.01742554,  0.02102661, ..., -0.01409912,
         -0.01385498, -0.01309204]),
  'sampling_rate': 48000},
 'labels': [1, 0, 0, 0, 0, 0]}

In [67]:
from whoot_model_training.whoot_model_training.trainer import WhootTrainer, WhootTrainingArguments
from whoot_model_training.whoot_model_training.data_extractor import buowset_extractor
from whoot_model_training.whoot_model_training.models import TimmModel, TimmInputs, TimmModelConfig
from whoot_model_training.whoot_model_training import CometMLLoggerSupplement

from whoot_model_training.whoot_model_training.preprocessors import (
    MelModelInputPreprocessor
)

In [68]:
model = TimmModel.from_pretrained(
    "/home/sean/whoot/model_checkpoints/buowset1.1_efficientnet_b1_08_15_2025_13:16:04/checkpoint-1580/",
)
preprocessor = MelModelInputPreprocessor(
    TimmInputs, duration=3
)

/home/sean/whoot/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for conv_stem.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/home/sean/whoot/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model

In [69]:
model

TimmModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Con

In [70]:
# for audio_info in ds["train"]:
    
#     audio = audio_info["audio"]
#     audio_info["audio"] = [audio_info["audio"]]
#     audio_info["labels"] = torch.Tensor([0,0,0,0,0,0])
#     for i in range(0, audio["array"].shape[0], audio["sampling_rate"] * 3):
#         input = audio["array"][i:i + audio["sampling_rate"] * 3]
#         print(i/audio["sampling_rate"], audio_info["audio"][0]["path"], model(preprocessor(audio_info)))
#     break

ds["train"].set_transform(preprocessor)


In [71]:
torch.Tensor(np.concat(ds["train"][0:5].spectrogram)).shape

torch.Size([5, 256, 259])

In [72]:
ds["train"][0:5].labels

array([[1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0.]], dtype=float32)

In [102]:
import torch
import numpy as np

results = []
model.cuda()
for i in range(0, ds["test"].shape[0], 5):
    x = TimmInputs(labels=torch.Tensor(ds["test"][i:i+5].labels).cuda(),
        spectrogram=torch.Tensor(np.concat(ds["test"][i:i+5].spectrogram)).unsqueeze(1).cuda())
    out = model(x)
    results.append(np.array(torch.argmax(out["logits"], dim=1).cpu()))
    if i > 500:
        break


tensor([[-11.9801, -11.7510, -13.3179,  -3.5983, -12.7942,   3.7165],
        [-22.5569, -18.8616, -21.6555, -14.9558, -26.1693,  16.7887],
        [-28.0504, -23.7132, -26.2290, -16.7775, -32.3714,  18.8534],
        [-20.3040, -13.5685, -22.6947, -13.0654, -23.6788,  14.3298],
        [-20.7984, -16.2405, -23.5307, -12.0199, -25.2754,  13.7340]],
       device='cuda:0', grad_fn=<AddmmBackward0>) tensor(5.7054, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[-10.6463, -11.6919, -12.0322,  -9.4530, -15.9026,   8.7405],
        [-18.7760, -18.9497, -19.5558, -13.0709, -24.5872,  13.3310],
        [-17.6199, -15.1691, -16.6451, -13.0334, -22.5385,  12.1574],
        [ -9.9132, -10.7750, -12.4069,  -8.6010, -15.7670,   8.2736],
        [-15.9893, -15.6640, -14.2749, -10.5522, -20.2389,  11.0540]],
       device='cuda:0', grad_fn=<AddmmBackward0>) tensor(4.2167, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[-22.7153, -19

/tmp/ipykernel_817618/3868621331.py:10: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  results.append(np.array(torch.argmax(out["logits"], dim=1).cpu()))


tensor([[-25.6627, -19.1187, -26.0079, -17.6518, -27.8401,  17.9818],
        [-20.4639, -14.6373, -21.0470, -13.1323, -21.8942,  14.0545],
        [-17.6472, -12.9593, -16.8312, -13.0439, -17.8351,  12.1448],
        [-24.5938, -19.4026, -23.6713, -15.1207, -27.0474,  16.7659],
        [-23.4175, -18.7242, -20.0196, -14.1604, -26.2955,  15.2203]],
       device='cuda:0', grad_fn=<AddmmBackward0>) tensor(6.2651, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[-22.5956, -18.4819, -21.1396, -15.2815, -24.7799,  15.8402],
        [-22.8801, -18.1315, -20.3435, -14.7207, -23.3664,  14.7429],
        [-25.1979, -21.6955, -26.2941, -18.0929, -27.1179,  17.7363],
        [-27.9811, -22.1019, -29.0579, -17.6141, -32.1800,  18.6680],
        [-22.1971, -18.7451, -21.6321, -15.6149, -19.7802,  13.7537]],
       device='cuda:0', grad_fn=<AddmmBackward0>) tensor(6.7198, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor([[-21.9624, -18

In [103]:
og_ds = raw_audio_extractor("data/2023/Otay/5_seconds")

Resolving data files:   0%|          | 0/31537 [00:00<?, ?it/s]

In [104]:
detections = og_ds["test"][np.arange(510)[np.concat(results) != 5]]

In [105]:
for i in np.arange(510)[np.concat(results) != 5]:
    print(np.concat(results)[i])
    import IPython
    display(IPython.display.Audio(og_ds["test"][int(i)]["audio"]["path"]))

3


3


3


3


3


...

3


3


0


3


3


In [106]:
[np.arange(510)[np.concat(results) != 5]]

[array([ 66,  88,  94,  97, 114, 115, 116, 152, 188, 191, 194, 197, 284])]